# v28 — Isu #1: Kesadaran Support/Resistance (S/R Proximity Filter)

**Konteks (dari brief perbaikan Isu Signal v13, 3 isu total -- ini isu #1)**: robot v13 belum
sadar apakah harga saat sinyal muncul sedang dekat area Support/Resistance penting. Sinyal bisa
"benar" (searah trend, skor lolos threshold) tapi entry di ujung pergerakan dekat S/R
berlawanan -- rawan reversal/mantul sebelum breakout.

**Tujuan (sesuai brief)**: tambah kesadaran S/R dari H1 & M15 (bukan cuma M5). Kalau sinyal
muncul dekat S/R berlawanan arah, robot harus SKIP, atau tetap ORDER dengan SL/TP disesuaikan
kalau sinyal sangat kuat -- bukan filter biner buang-semua. Jarak "dekat" harus ATR-relative
(BUKAN poin fixed -- pelajaran dari kegagalan SL fixed v04 yang rusak saat harga naik ke $4300+).

**Modul yang sudah dibangun** (`app/utils/indicators/support_resistance.py`):
- `add_swing_pivots(df, lookback=5)`: deteksi pivot high/low sejati (butuh konfirmasi kiri DAN
  kanan candle, beda dari rolling max/min yang cuma lihat ke belakang spt bos_choch.py)
- `build_sr_levels(df, lookback=5, max_levels=5)`: level resistance/support TERDEKAT di
  atas/bawah close saat ini, dari 5 pivot terakhir yang sudah CONFIRMED (no-lookahead
  divalidasi: hasil di candle manapun IDENTIK baik dihitung dari data penuh maupun data yang
  dipotong setelah candle itu).

**Yang dicari di notebook ini (SEMUA lewat backtest, TIDAK ditebak, sesuai batasan brief)**:
1. Cek pola dulu: apakah trade yang entry dekat S/R berlawanan memang lebih sering rugi?
2. Threshold jarak "dekat" dalam satuan ATR (kandidat: 0.5x/1x/1.5x/2x/3x)
3. Sumber S/R mana yang lebih berguna: H1 saja, M15 saja, atau kombinasi (paling ketat dari
   keduanya)
4. Strategi saat dekat S/R: SKIP total vs tetap ORDER dengan SL/TP disesuaikan (TP diperkecil
   krn ruang sempit) -- keputusan bertingkat, bukan biner
5. Ablation test: dampak filter INI SAJA (dengan vs tanpa), out-of-sample

**Metodologi**: TRAIN (2019-2023) / TEST (2024-2026) split, walk-forward, spread real 1.82,
v12_score ASLI + Order Block filter + H1 alignment (v13 sesungguhnya, bukan reimplementasi),
kriteria kejujuran (kandidat harus MENGUNGGULI baseline v13 murni PF di TRAIN *dan* TEST).

**TIDAK ADA perubahan ke `usecase.py`** sampai hasil ini divalidasi & disetujui terpisah.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STRATEGY_NAME = "m5_scalping"
VERSION = "v28"

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed" / STRATEGY_NAME
EXPORT_DIR = PROJECT_ROOT / "dataset" / "exports" / STRATEGY_NAME / VERSION
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
(PROCESSED_DIR / VERSION).mkdir(parents=True, exist_ok=True)

INITIAL_EQUITY = 100.0
RISK_PCT = 0.01
CONTRACT_SIZE = 100.0
MIN_LOT = 0.01
LOT_STEP = 0.01
REAL_SPREAD = 1.82

MIN_SAMPLE_TRAIN = 30
MIN_SAMPLE_TEST = 15

pd.set_option("display.width", 180)
plt.rcParams["figure.figsize"] = (14, 5)

## 1. Load cache v23 (skor v12 + OB + H1 EMA, 2019-2026) + tambah level S/R dari H1 & M15

In [2]:
SR_CACHE_PATH = PROCESSED_DIR / VERSION / "df_2019_2026_sr.parquet"

if SR_CACHE_PATH.exists():
    print(f"Load dari cache: {SR_CACHE_PATH}")
    df = pd.read_parquet(SR_CACHE_PATH)
else:
    print("Belum ada cache -- load v23 base + hitung & merge level S/R H1 & M15...")
    from app.utils.indicators.support_resistance import build_sr_levels

    v23_cache = PROCESSED_DIR / "v23" / "df_2019_2026_full_mtf.parquet"
    assert v23_cache.exists(), "Cache v23 belum ada"
    df = pd.read_parquet(v23_cache)

    for tf_file, tf_delta, prefix in [("h1", pd.Timedelta(hours=1), "h1"), ("m15", pd.Timedelta(minutes=15), "m15")]:
        df_tf = pd.read_csv(
            PROCESSED_DIR / "v01" / f"xauusd_{tf_file}_full_indicators.csv",
            usecols=["datetime", "open", "high", "low", "close"],
        )
        df_tf["datetime"] = pd.to_datetime(df_tf["datetime"])
        df_tf = df_tf.sort_values("datetime").reset_index(drop=True)
        df_tf = build_sr_levels(df_tf)
        # available_at = waktu candle tf ini SELESAI (close), supaya M5 di jam yg sama TIDAK
        # "mengintip" candle tf yang belum closed -- pola sama persis dgn merge H1 EMA yg sudah ada.
        df_tf["available_at"] = df_tf["datetime"] + tf_delta
        df_tf = df_tf.rename(columns={"sr_resistance": f"{prefix}_sr_resistance", "sr_support": f"{prefix}_sr_support"})
        cols = ["available_at", f"{prefix}_sr_resistance", f"{prefix}_sr_support"]
        df = pd.merge_asof(
            df.sort_values("datetime"), df_tf[cols].sort_values("available_at"),
            left_on="datetime", right_on="available_at", direction="backward",
        )
        df = df.drop(columns=["available_at"])
        print(f"  merged S/R {prefix.upper()}")

    df.to_parquet(SR_CACHE_PATH, index=False)
    print(f"Tersimpan ke cache: {SR_CACHE_PATH}")

print(f"\nTotal candle: {len(df)}, {df['datetime'].min()} -> {df['datetime'].max()}")
print(f"Kolom S/R: {[c for c in df.columns if 'sr_' in c]}")

Load dari cache: D:\Projects\robot-scalping\dataset\processed\m5_scalping\v28\df_2019_2026_sr.parquet

Total candle: 518403, 2019-01-01 23:00:00+00:00 -> 2026-08-06 12:35:00+00:00
Kolom S/R: ['h1_sr_resistance', 'h1_sr_support', 'm15_sr_resistance', 'm15_sr_support']


## 2. Backtest engine v28: v13 + filter S/R proximity (skip / SL-TP adjusted / normal)

In [3]:
def check_h1_alignment_v28(h1_ema_50, h1_ema_200, direction: str) -> bool:
    if h1_ema_50 is None or h1_ema_200 is None or not np.isfinite(h1_ema_50) or not np.isfinite(h1_ema_200):
        return True
    h1_trend = "UP" if h1_ema_50 > h1_ema_200 else ("DOWN" if h1_ema_50 < h1_ema_200 else "FLAT")
    if direction == "BUY" and h1_trend == "DOWN":
        return False
    if direction == "SELL" and h1_trend == "UP":
        return False
    return True


def run_backtest_v28(
    df_signals: pd.DataFrame,
    adx_min: float = 18.0,
    min_signal_score: float = 9.0,
    sl_mult: float = 2.0,
    tp_mult: float = 4.0,
    max_hold: int = 12,
    sr_source: str = "none",           # "none", "h1", "m15", "both" (both = level TERDEKAT dari kedua tf)
    sr_near_atr_mult: float = 1.0,     # jarak "dekat" S/R berlawanan, dlm kelipatan ATR
    sr_action: str = "skip",           # "skip" atau "adjust" (TP dipangkas ke level S/R)
    sr_strong_score_bonus: float = 0.0,  # kalau skor >= min_signal_score + bonus ini, filter S/R diabaikan (sinyal "sangat kuat")
    sr_min_atr_for_breakout: float = None,  # None=nonaktif. Kalau diisi: izinkan order dekat S/R
                                             # SELAMA atr >= nilai ini (candle "bertenaga", indikasi breakout),
                                             # meski TIDAK jauh & TIDAK skor kuat -- dari temuan investigasi
                                             # ad-hoc: trade TEMBUS py ATR mean 1.880, trade MANTUL py ATR mean
                                             # 1.149 (p=0.0000, signifikan) -- jarak S/R & skor TIDAK signifikan.
    require_ob_filter: bool = True,
    require_h1_alignment: bool = True,
    spread_points: float = REAL_SPREAD,
    use_fixed_lot: bool = False,  # True -> lot TETAP (nilai dari fixed_lot_value), TIDAK bergantung equity
    fixed_lot_value: float = MIN_LOT,  # nilai lot dipakai kalau use_fixed_lot=True (default 0.01)
    initial_equity_override: float = None,  # None -> pakai INITIAL_EQUITY global; kalau diisi, basis modal beda (mis. $2000)
    max_total_dd_pct: float = None,  # None=nonaktif. Kalau diisi (mis. 40.0): SIMULASI KILL-SWITCH
                                       # TOTAL spt robot live -- begitu equity turun dari PEAK
                                       # melebihi persentase ini, trading PAUSE PERMANEN sampai
                                       # akhir data (TIDAK auto-resume, sama persis desain live
                                       # -- lihat MAX_TOTAL_DRAWDOWN_PCT di .env & _check_drawdown_guard()).
                                       # WAJIB dipakai utk full period 2019-2026 tanpa ini, modal
                                       # 00 + lot fixed bisa jatuh negatif krn tidak ada rem sama
                                       # sekali -- itu BUKAN skenario realistis (robot live py rem).
) -> pd.DataFrame:
    close_arr = df_signals["close"].to_numpy()
    high_arr = df_signals["high"].to_numpy()
    low_arr = df_signals["low"].to_numpy()
    adx_arr = df_signals["adx"].to_numpy()
    atr_arr = df_signals["atr"].to_numpy()
    score_arr = df_signals["v12_score"].to_numpy()
    ob_bull_arr = df_signals["ob_bull"].to_numpy()
    ob_bear_arr = df_signals["ob_bear"].to_numpy()
    h1_ob_bull_arr = df_signals["h1_ob_bull"].to_numpy()
    h1_ob_bear_arr = df_signals["h1_ob_bear"].to_numpy()
    h1_ema_50_arr = df_signals["h1_ema_50"].to_numpy()
    h1_ema_200_arr = df_signals["h1_ema_200"].to_numpy()
    h1_res_arr = df_signals["h1_sr_resistance"].to_numpy()
    h1_sup_arr = df_signals["h1_sr_support"].to_numpy()
    m15_res_arr = df_signals["m15_sr_resistance"].to_numpy()
    m15_sup_arr = df_signals["m15_sr_support"].to_numpy()
    datetime_arr = df_signals["datetime"].to_numpy()
    n = len(df_signals)

    trades = []
    base_equity = initial_equity_override if initial_equity_override is not None else INITIAL_EQUITY
    equity = base_equity
    peak_equity = base_equity
    total_dd_paused = False
    i = 0
    while i < n:
        adx, atr, close, score = adx_arr[i], atr_arr[i], close_arr[i], score_arr[i]
        if not np.isfinite(atr) or atr <= 0 or not np.isfinite(adx) or not np.isfinite(score):
            i += 1
            continue
        if adx < adx_min:
            i += 1
            continue

        # Kill-switch TOTAL (spt _check_drawdown_guard live) -- PERMANEN, TIDAK auto-resume.
        if max_total_dd_pct is not None:
            if not total_dd_paused and peak_equity > 0:
                current_dd_pct = (peak_equity - equity) / peak_equity * 100
                if current_dd_pct >= max_total_dd_pct:
                    total_dd_paused = True
            if total_dd_paused:
                i += 1
                continue

        direction = None
        if score >= min_signal_score:
            direction = "BUY"
        elif score <= -min_signal_score:
            direction = "SELL"
        if direction is None:
            i += 1
            continue

        if require_ob_filter:
            opposing_ob = (
                (direction == "BUY" and (ob_bear_arr[i] > 0 or h1_ob_bear_arr[i] > 0)) or
                (direction == "SELL" and (ob_bull_arr[i] > 0 or h1_ob_bull_arr[i] > 0))
            )
            if opposing_ob:
                i += 1
                continue
        if require_h1_alignment:
            if not check_h1_alignment_v28(h1_ema_50_arr[i], h1_ema_200_arr[i], direction):
                i += 1
                continue

        sl_points = sl_mult * atr
        tp_points = tp_mult * atr
        entry_price = close + (spread_points if direction == "BUY" else -spread_points)
        tp_price = entry_price + tp_points if direction == "BUY" else entry_price - tp_points
        sl_price = entry_price - sl_points if direction == "BUY" else entry_price + sl_points
        mode = "NORMAL"

        # --- Filter S/R proximity ---
        if sr_source != "none":
            # Level BERLAWANAN arah sinyal: BUY -> resistance di atas; SELL -> support di bawah
            opposing_level = None
            if direction == "BUY":
                candidates = []
                if sr_source in ("h1", "both") and np.isfinite(h1_res_arr[i]):
                    candidates.append(h1_res_arr[i])
                if sr_source in ("m15", "both") and np.isfinite(m15_res_arr[i]):
                    candidates.append(m15_res_arr[i])
                if candidates:
                    opposing_level = min(candidates)  # resistance TERDEKAT (paling ketat)
            else:
                candidates = []
                if sr_source in ("h1", "both") and np.isfinite(h1_sup_arr[i]):
                    candidates.append(h1_sup_arr[i])
                if sr_source in ("m15", "both") and np.isfinite(m15_sup_arr[i]):
                    candidates.append(m15_sup_arr[i])
                if candidates:
                    opposing_level = max(candidates)

            if opposing_level is not None:
                dist_to_level = abs(opposing_level - close)
                is_near = dist_to_level <= (sr_near_atr_mult * atr)
                is_strong_signal = abs(score) >= (min_signal_score + sr_strong_score_bonus)

                is_breakout_atr = sr_min_atr_for_breakout is not None and atr >= sr_min_atr_for_breakout
                if is_near and not is_strong_signal and not is_breakout_atr:
                    if sr_action == "skip":
                        i += 1
                        continue
                    elif sr_action == "adjust":
                        # TP dipangkas ke level S/R (dgn sedikit buffer 0.1x ATR spy tidak persis di level)
                        buffer = 0.1 * atr
                        adjusted_tp = opposing_level - buffer if direction == "BUY" else opposing_level + buffer
                        # Cuma pangkas kalau adjusted_tp itu LEBIH DEKAT dari TP normal (jangan perlebar)
                        if direction == "BUY" and adjusted_tp < tp_price and adjusted_tp > entry_price:
                            tp_price = adjusted_tp
                            mode = "SR_ADJUSTED"
                        elif direction == "SELL" and adjusted_tp > tp_price and adjusted_tp < entry_price:
                            tp_price = adjusted_tp
                            mode = "SR_ADJUSTED"

        entry_time = datetime_arr[i]
        exit_price = None
        exit_idx = min(i + max_hold, n - 1)
        window_end = min(i + 1 + max_hold, n)
        for candle_idx in range(i + 1, window_end):
            c_high, c_low = high_arr[candle_idx], low_arr[candle_idx]
            hit_tp = c_high >= tp_price if direction == "BUY" else c_low <= tp_price
            hit_sl = c_low <= sl_price if direction == "BUY" else c_high >= sl_price
            if hit_sl:
                exit_price, exit_time = sl_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
            if hit_tp:
                exit_price, exit_time = tp_price, datetime_arr[candle_idx]
                exit_idx = candle_idx
                break
        if exit_price is None:
            exit_price, exit_time = close_arr[exit_idx], datetime_arr[exit_idx]

        next_i = exit_idx + 1
        price_move = (exit_price - entry_price) if direction == "BUY" else (entry_price - exit_price)
        if use_fixed_lot:
            lot = fixed_lot_value
        else:
            risk_amount = equity * RISK_PCT
            lot = max(round(math.floor((risk_amount / (sl_points * CONTRACT_SIZE)) / LOT_STEP) * LOT_STEP, 2), MIN_LOT) if sl_points > 0 and risk_amount > 0 else MIN_LOT
        pnl = price_move * lot * CONTRACT_SIZE
        equity += pnl
        peak_equity = max(peak_equity, equity)
        trades.append({
            "entry_time": entry_time, "mode": mode, "direction": direction, "pnl": pnl,
            "result": "WIN" if pnl > 0 else "LOSS", "equity_after": equity,
        })
        i = next_i

    return pd.DataFrame(trades)


def evaluate(trades: pd.DataFrame, initial_equity: float) -> dict:
    if trades.empty:
        return {"total_trades": 0, "win_rate_pct": 0, "profit_factor": 0, "net_pnl": 0, "max_drawdown_pct": 0}
    wins = trades[trades["pnl"] > 0]
    losses = trades[trades["pnl"] <= 0]
    gross_profit = wins["pnl"].sum()
    gross_loss = losses["pnl"].sum()
    equity_series = pd.Series([initial_equity] + trades["equity_after"].tolist())
    running_max = equity_series.cummax()
    drawdown = (equity_series - running_max) / running_max * 100
    return {
        "total_trades": len(trades),
        "win_rate_pct": round(len(wins) / len(trades) * 100, 2),
        "profit_factor": round(gross_profit / abs(gross_loss), 2) if gross_loss != 0 else float("inf"),
        "net_pnl": round(gross_profit + gross_loss, 2),
        "max_drawdown_pct": round(drawdown.min(), 2),
    }

print("Backtest engine v28 siap.")

Backtest engine v28 siap.


## 3. TRAIN/TEST split & Baseline (v13 murni, sr_source='none')

In [4]:
TRAIN_END = pd.Timestamp("2024-01-01", tz="UTC")
df_train = df[df["datetime"] < TRAIN_END].reset_index(drop=True)
df_test = df[df["datetime"] >= TRAIN_END].reset_index(drop=True)
print(f"TRAIN (2019-2023): {len(df_train)} candle | TEST (2024-2026): {len(df_test)} candle")

trades_base_train = run_backtest_v28(df_train, sr_source="none")
trades_base_test = run_backtest_v28(df_test, sr_source="none")
baseline_train = evaluate(trades_base_train, INITIAL_EQUITY)
baseline_test = evaluate(trades_base_test, INITIAL_EQUITY)
print("=== Baseline: v13 murni (tanpa filter S/R) ===")
print("TRAIN:", baseline_train)
print("TEST :", baseline_test)

TRAIN (2019-2023): 351136 candle | TEST (2024-2026): 167267 candle


=== Baseline: v13 murni (tanpa filter S/R) ===
TRAIN: {'total_trades': 2282, 'win_rate_pct': 15.69, 'profit_factor': np.float64(0.38), 'net_pnl': np.float64(-2348.28), 'max_drawdown_pct': np.float64(-2348.28)}
TEST : {'total_trades': 1105, 'win_rate_pct': 45.7, 'profit_factor': np.float64(1.58), 'net_pnl': np.float64(1588.28), 'max_drawdown_pct': np.float64(-229.55)}


## 4. Cek pola dulu: apakah trade WIN vs LOSS beda jarak ke S/R saat entry?

Sebelum grid search -- baca dulu: apakah trade yang entry DEKAT S/R berlawanan arah memang
lebih sering rugi (hipotesis brief), sesuai pola v27 (cek dulu sebelum tuning parameter).

In [5]:
trades_base_train_idx = trades_base_train.copy()
trades_base_train_idx["entry_time"] = pd.to_datetime(trades_base_train_idx["entry_time"])
df_train_lookup = df_train[["datetime", "close", "atr", "h1_sr_resistance", "h1_sr_support", "m15_sr_resistance", "m15_sr_support"]].rename(columns={"datetime": "entry_time"})
trades_with_sr = trades_base_train_idx.merge(df_train_lookup, on="entry_time", how="left")

def dist_to_opposing_level_atr(row, tf_prefix):
    res = row[f"{tf_prefix}_sr_resistance"]
    sup = row[f"{tf_prefix}_sr_support"]
    level = res if row["direction"] == "BUY" else sup
    if pd.isna(level) or row["atr"] <= 0:
        return np.nan
    return abs(level - row["close"]) / row["atr"]

for tf in ["h1", "m15"]:
    trades_with_sr[f"{tf}_dist_atr"] = trades_with_sr.apply(lambda r: dist_to_opposing_level_atr(r, tf), axis=1)

print("=== Jarak ke level S/R berlawanan (dalam x ATR) -- WIN vs LOSS ===")
for tf in ["h1", "m15"]:
    col = f"{tf}_dist_atr"
    win_med = trades_with_sr[trades_with_sr["result"]=="WIN"][col].median()
    loss_med = trades_with_sr[trades_with_sr["result"]=="LOSS"][col].median()
    has_level_pct = trades_with_sr[col].notna().mean() * 100
    print(f"{tf.upper()}: WIN median={win_med:.2f}x ATR, LOSS median={loss_med:.2f}x ATR "
          f"(trade dgn level teridentifikasi: {has_level_pct:.1f}%)")

print()
print("=== Win rate berdasar bucket jarak H1 S/R (x ATR) ===")
trades_with_sr["h1_dist_bucket"] = pd.cut(trades_with_sr["h1_dist_atr"], bins=[0,0.5,1.0,1.5,2.0,3.0,100], labels=["<0.5x","0.5-1x","1-1.5x","1.5-2x","2-3x",">3x"])
for bucket, g in trades_with_sr.groupby("h1_dist_bucket", observed=True):
    wr = (g["result"]=="WIN").mean()*100
    print(f"  {bucket}: n={len(g)}, win_rate={wr:.1f}%")

=== Jarak ke level S/R berlawanan (dalam x ATR) -- WIN vs LOSS ===
H1: WIN median=2.32x ATR, LOSS median=2.83x ATR (trade dgn level teridentifikasi: 80.5%)
M15: WIN median=1.40x ATR, LOSS median=1.27x ATR (trade dgn level teridentifikasi: 78.4%)

=== Win rate berdasar bucket jarak H1 S/R (x ATR) ===
  <0.5x: n=214, win_rate=17.8%
  0.5-1x: n=187, win_rate=16.6%
  1-1.5x: n=186, win_rate=17.7%
  1.5-2x: n=144, win_rate=13.2%
  2-3x: n=239, win_rate=16.3%
  >3x: n=866, win_rate=12.6%


## 4b. Investigasi mendalam: dari trade DEKAT S/R, mana yang MANTUL (rugi) vs TEMBUS
(untung), dan apakah TP dipangkas bisa menyelamatkan yang mantul?

**Pertanyaan user**: (1) kalau order dibiarkan jalan di area S/R, apa benar banyak yang
mantul & rugi? (2) kalau ada yang tembus, alasan/validasinya apa yang kuat (bukan sekadar
"kadang menang kadang kalah")? (3) untuk yang MANTUL, apakah TP dipangkas (SL/TP adjusted)
bisa menyelamatkan trade itu -- apakah harga sempat mendekati target sebelum berbalik?

**Metodologi**: ambil SEMUA sinyal v13 yang lolos filter dasar (skor, OB, H1 alignment) TANPA
filter S/R apapun, fokus ke yang entry dekat S/R H1 (<=2.0x ATR, threshold dari kandidat
terbaik Section 5-6). Bandingkan trade yang MANTUL (LOSS) vs TEMBUS (WIN) di 4 faktor: jarak
ke S/R H1, jarak ke S/R M15, kekuatan skor sinyal, dan ATR (volatilitas) saat entry -- uji
statistik (Mann-Whitney U) utk tau mana yang beneran signifikan, bukan cuma kelihatan beda.

In [6]:
from scipy.stats import mannwhitneyu

def get_all_signals_with_sr(df_signals, adx_min=18.0, min_signal_score=9.0, sl_mult=2.0, tp_mult=4.0, max_hold=12,
                              require_ob_filter=True, require_h1_alignment=True, spread_points=REAL_SPREAD):
    """SEMUA sinyal v13 (TANPA filter S/R apapun) + info jarak ke S/R, utk analisis pola
    (dipisah dari run_backtest_v28 yang dipakai grid search -- ini murni observasional)."""
    close_arr = df_signals["close"].to_numpy()
    high_arr = df_signals["high"].to_numpy()
    low_arr = df_signals["low"].to_numpy()
    adx_arr = df_signals["adx"].to_numpy()
    atr_arr = df_signals["atr"].to_numpy()
    score_arr = df_signals["v12_score"].to_numpy()
    ob_bull_arr = df_signals["ob_bull"].to_numpy()
    ob_bear_arr = df_signals["ob_bear"].to_numpy()
    h1_ob_bull_arr = df_signals["h1_ob_bull"].to_numpy()
    h1_ob_bear_arr = df_signals["h1_ob_bear"].to_numpy()
    h1_ema_50_arr = df_signals["h1_ema_50"].to_numpy()
    h1_ema_200_arr = df_signals["h1_ema_200"].to_numpy()
    h1_res_arr = df_signals["h1_sr_resistance"].to_numpy()
    h1_sup_arr = df_signals["h1_sr_support"].to_numpy()
    m15_res_arr = df_signals["m15_sr_resistance"].to_numpy()
    m15_sup_arr = df_signals["m15_sr_support"].to_numpy()
    datetime_arr = df_signals["datetime"].to_numpy()
    n = len(df_signals)

    results = []
    i = 0
    while i < n:
        adx, atr, close, score = adx_arr[i], atr_arr[i], close_arr[i], score_arr[i]
        if not np.isfinite(atr) or atr <= 0 or not np.isfinite(adx) or not np.isfinite(score):
            i += 1
            continue
        if adx < adx_min:
            i += 1
            continue
        direction = None
        if score >= min_signal_score:
            direction = "BUY"
        elif score <= -min_signal_score:
            direction = "SELL"
        if direction is None:
            i += 1
            continue
        if require_ob_filter:
            opposing_ob = (
                (direction == "BUY" and (ob_bear_arr[i] > 0 or h1_ob_bear_arr[i] > 0)) or
                (direction == "SELL" and (ob_bull_arr[i] > 0 or h1_ob_bull_arr[i] > 0))
            )
            if opposing_ob:
                i += 1
                continue
        if require_h1_alignment:
            if not check_h1_alignment_v28(h1_ema_50_arr[i], h1_ema_200_arr[i], direction):
                i += 1
                continue

        sl_points = sl_mult * atr
        tp_points = tp_mult * atr
        entry_price = close + (spread_points if direction == "BUY" else -spread_points)
        tp_price = entry_price + tp_points if direction == "BUY" else entry_price - tp_points
        sl_price = entry_price - sl_points if direction == "BUY" else entry_price + sl_points

        opposing_h1 = h1_res_arr[i] if direction == "BUY" else h1_sup_arr[i]
        opposing_m15 = m15_res_arr[i] if direction == "BUY" else m15_sup_arr[i]
        dist_h1_atr = abs(opposing_h1 - close) / atr if np.isfinite(opposing_h1) else np.nan
        dist_m15_atr = abs(opposing_m15 - close) / atr if np.isfinite(opposing_m15) else np.nan

        entry_time = datetime_arr[i]
        exit_price = None
        exit_idx = min(i + max_hold, n - 1)
        window_end = min(i + 1 + max_hold, n)
        mfe_price = entry_price
        for candle_idx in range(i + 1, window_end):
            c_high, c_low = high_arr[candle_idx], low_arr[candle_idx]
            mfe_price = max(mfe_price, c_high) if direction == "BUY" else min(mfe_price, c_low)
            hit_tp = c_high >= tp_price if direction == "BUY" else c_low <= tp_price
            hit_sl = c_low <= sl_price if direction == "BUY" else c_high >= sl_price
            if hit_sl:
                exit_price, exit_reason = sl_price, "SL"
                exit_idx = candle_idx
                break
            if hit_tp:
                exit_price, exit_reason = tp_price, "TP"
                exit_idx = candle_idx
                break
        if exit_price is None:
            exit_price, exit_reason = close_arr[exit_idx], "TIMEOUT"

        result = "WIN" if (exit_price - entry_price) * (1 if direction == "BUY" else -1) > 0 else "LOSS"
        opposing_level = opposing_h1
        mfe_move = (mfe_price - entry_price) if direction == "BUY" else (entry_price - mfe_price)
        sr_target_dist = abs(opposing_level - entry_price) if np.isfinite(opposing_level) else np.nan

        results.append({
            "entry_time": entry_time, "direction": direction, "result": result, "exit_reason": exit_reason,
            "dist_h1_atr": dist_h1_atr, "dist_m15_atr": dist_m15_atr, "score": score, "atr": atr,
            "mfe_move": mfe_move, "sr_target_dist": sr_target_dist,
        })
        i = exit_idx + 1

    return pd.DataFrame(results)


all_signals_sr = get_all_signals_with_sr(df_train)
print(f"Total sinyal v13 (tanpa filter S/R): {len(all_signals_sr)}")

near_h1 = all_signals_sr[all_signals_sr["dist_h1_atr"] <= 2.0].copy()
bounce = near_h1[near_h1["result"] == "LOSS"]
breakout = near_h1[near_h1["result"] == "WIN"]
print(f"\n=== Dari {len(near_h1)} trade DEKAT S/R H1 (<=2.0x ATR) ===")
print(f"MANTUL (LOSS): {len(bounce)} ({len(bounce)/len(near_h1)*100:.1f}%)")
print(f"TEMBUS (WIN): {len(breakout)} ({len(breakout)/len(near_h1)*100:.1f}%)")

print("\n=== Pembeda MANTUL vs TEMBUS (uji Mann-Whitney U) ===")
for col in ["dist_h1_atr", "dist_m15_atr", "score", "atr"]:
    b, t = bounce[col].dropna(), breakout[col].dropna()
    b_mean, t_mean = b.mean(), t.mean()
    if len(b) > 5 and len(t) > 5:
        _, p = mannwhitneyu(b, t)
        flag = " <-- SIGNIFIKAN (p<0.05)" if p < 0.05 else ""
        print(f"  {col}: MANTUL mean={b_mean:.3f}, TEMBUS mean={t_mean:.3f}, p={p:.4f}{flag}")

print("\n=== MFE (Max Favorable Excursion) trade MANTUL -- apakah sempat dekat target S/R sblm berbalik? ===")
mantul_pct_of_sr = (bounce["mfe_move"] / bounce["sr_target_dist"] * 100).clip(upper=200)
bins = [-1000, 0, 25, 50, 75, 90, 100, 1000]
labels = ["negatif/mundur langsung", "0-25%", "25-50%", "50-75%", "75-90%", "90-100%", ">100% (lewat S/R tapi ttp LOSS)"]
bucket = pd.cut(mantul_pct_of_sr, bins=bins, labels=labels)
print(bucket.value_counts().sort_index())
reached_90pct = (mantul_pct_of_sr >= 90).sum()
print(f"\nKesimpulan: dari {len(bounce)} trade MANTUL, cuma {reached_90pct} ({reached_90pct/len(bounce)*100:.1f}%) "
      f"yang SEMPAT mencapai >=90% jarak ke S/R sblm berbalik -- TP dipangkas TIDAK akan menyelamatkan "
      f"mayoritas trade yg mantul, krn harga memang salah arah SEJAK AWAL (mayoritas di bucket 'negatif/mundur').")

Total sinyal v13 (tanpa filter S/R): 2282

=== Dari 733 trade DEKAT S/R H1 (<=2.0x ATR) ===
MANTUL (LOSS): 612 (83.5%)
TEMBUS (WIN): 121 (16.5%)

=== Pembeda MANTUL vs TEMBUS (uji Mann-Whitney U) ===
  dist_h1_atr: MANTUL mean=0.919, TEMBUS mean=0.882, p=0.5549
  dist_m15_atr: MANTUL mean=1.645, TEMBUS mean=1.309, p=0.8775
  score: MANTUL mean=2.012, TEMBUS mean=0.481, p=0.5590
  atr: MANTUL mean=1.149, TEMBUS mean=1.880, p=0.0000 <-- SIGNIFIKAN (p<0.05)

=== MFE (Max Favorable Excursion) trade MANTUL -- apakah sempat dekat target S/R sblm berbalik? ===
negatif/mundur langsung            471
0-25%                               30
25-50%                              31
50-75%                              22
75-90%                               7
90-100%                              2
>100% (lewat S/R tapi ttp LOSS)     49
Name: count, dtype: int64

Kesimpulan: dari 612 trade MANTUL, cuma 51 (8.3%) yang SEMPAT mencapai >=90% jarak ke S/R sblm berbalik -- TP dipangkas TIDAK akan menyelama

## 4c. Grid search v2: filter S/R + syarat ATR minimum (pengganti 'adjust', yg terbukti
tidak efektif di 4b)

**Temuan kunci dari 4b**: ATR adalah SATU-SATUNYA faktor yang signifikan (p=0.0000)
membedakan trade TEMBUS (ATR mean 1.880) vs MANTUL (ATR mean 1.149) -- bukan jarak ke S/R,
bukan kekuatan skor. Dan 'adjust' (TP dipangkas) TERBUKTI tidak efektif krn 91.7% trade
mantul harganya salah arah SEJAK AWAL, TP sekecil apapun tidak akan menyelamatkan.

**Hipotesis baru utk digrid search**: drpd skip SEMUA yang dekat S/R, izinkan order KALAU
ATR saat itu >= threshold tertentu (candle "bertenaga", indikasi breakout beneran, bukan
cuma menyentuh level pelan-pelan). Ini `sr_min_atr_for_breakout` -- dikombinasikan dgn
parameter dari grid search Section 5-6 sebelumnya. **Threshold ATR dicari dari TRAIN,
divalidasi ketat di TEST -- supaya TIDAK overfitting** (bukan angka yang ditebak/dipas-paskan
ke hasil kelihatan bagus).

In [7]:
import itertools
import time as _time

GRID_V2 = {
    "sr_source": ["h1", "both"],
    "sr_near_atr_mult": [1.5, 2.0, 3.0],
    "sr_action": ["skip"],  # 'adjust' sudah terbukti tidak efektif di 4b, dikeluarkan dari grid
    "sr_strong_score_bonus": [0.0, 2.0, 4.0],
    "sr_min_atr_for_breakout": [None, 1.3, 1.5, 1.7, 1.9, 2.1],
}

combos_v2 = list(itertools.product(*GRID_V2.values()))
print(f"Total kombinasi grid v2: {len(combos_v2)}")

t0 = _time.time()
grid_v2_results = []
for idx, combo in enumerate(combos_v2):
    params = dict(zip(GRID_V2.keys(), combo))
    trades = run_backtest_v28(df_train, **params)
    metrics = evaluate(trades, INITIAL_EQUITY)
    metrics.update(params)
    grid_v2_results.append(metrics)
    if (idx + 1) % 30 == 0:
        print(f"  [{idx+1}/{len(combos_v2)}] {_time.time()-t0:.0f}s")

grid_v2_df = pd.DataFrame(grid_v2_results)
print(f"\nGrid search v2 selesai dalam {_time.time()-t0:.0f}s")

grid_v2_valid = grid_v2_df[grid_v2_df["total_trades"] >= MIN_SAMPLE_TRAIN].sort_values("profit_factor", ascending=False)
print(f"\n=== Top 20 kandidat v2 (sample TRAIN >= {MIN_SAMPLE_TRAIN}) ===")
print(grid_v2_valid.head(20).to_string(index=False))

beating_v2 = grid_v2_valid[grid_v2_valid["profit_factor"] > baseline_train["profit_factor"]]
print(f"\nBaseline TRAIN: PF={baseline_train['profit_factor']}")
print(f"Kandidat v2 mengungguli baseline TRAIN: {len(beating_v2)} dari {len(grid_v2_valid)}")

Total kombinasi grid v2: 108


  [30/108] 26s


  [60/108] 53s


  [90/108] 80s



Grid search v2 selesai dalam 96s

=== Top 20 kandidat v2 (sample TRAIN >= 30) ===
 total_trades  win_rate_pct  profit_factor  net_pnl  max_drawdown_pct sr_source  sr_near_atr_mult sr_action  sr_strong_score_bonus  sr_min_atr_for_breakout
         1149         25.50           0.63  -756.47           -756.47      both               3.0      skip                    4.0                      1.5
          989         24.17           0.61  -684.66           -684.66      both               3.0      skip                    4.0                      1.9
         1260         25.63           0.61  -867.51           -867.51      both               3.0      skip                    4.0                      1.3
         1054         24.48           0.61  -735.90           -735.90      both               3.0      skip                    4.0                      1.7
          944         23.20           0.60  -674.81           -674.81      both               3.0      skip                    4.0       

## 4d. Validasi TEST out-of-sample utk grid search v2 (kriteria kejujuran sama spt Section 6)

In [8]:
candidates_v2_passing = beating_v2.head(20)
print(f"Kandidat v2 TRAIN mengungguli baseline: {len(candidates_v2_passing)}")

if len(candidates_v2_passing) == 0:
    print("\n>>> TIDAK ADA kandidat v2 mengungguli baseline di TRAIN. Validasi TEST DIBATALKAN.")
else:
    test_v2_results = []
    for _, row in candidates_v2_passing.iterrows():
        params = {k: row[k] for k in GRID_V2.keys()}
        if pd.isna(params["sr_min_atr_for_breakout"]):
            params["sr_min_atr_for_breakout"] = None
        trades_test = run_backtest_v28(df_test, **params)
        m_test = evaluate(trades_test, INITIAL_EQUITY)
        test_v2_results.append({**params, "train_pf": row["profit_factor"], "train_n": row["total_trades"],
                                 "test_pf": m_test["profit_factor"], "test_n": m_test["total_trades"],
                                 "test_wr": m_test["win_rate_pct"], "test_netpnl": m_test["net_pnl"],
                                 "test_maxdd": m_test["max_drawdown_pct"]})

    test_v2_df = pd.DataFrame(test_v2_results)
    print("\n=== Validasi TEST kandidat v2 (yang menang di TRAIN) ===")
    print(test_v2_df.to_string(index=False))

    print(f"\nBaseline TEST: PF={baseline_test['profit_factor']}, net_pnl={baseline_test['net_pnl']}, "
          f"max_dd={baseline_test['max_drawdown_pct']}, n={baseline_test['total_trades']}")

    robust_v2 = test_v2_df[(test_v2_df["test_pf"] > baseline_test["profit_factor"]) & (test_v2_df["test_n"] >= MIN_SAMPLE_TEST)]
    print(f"\n>>> Kandidat v2 ROBUST (unggul TRAIN & TEST vs baseline, sample TEST>={MIN_SAMPLE_TEST}): {len(robust_v2)}")
    if len(robust_v2) > 0:
        print(robust_v2.to_string(index=False))
        print(f"\nDibanding grid search v1 (Section 5-6, tanpa syarat ATR): terbaik PF TEST 1.78")
        print(f"Grid search v2 (dgn syarat ATR breakout) terbaik PF TEST: {robust_v2['test_pf'].max():.2f}")

Kandidat v2 TRAIN mengungguli baseline: 20



=== Validasi TEST kandidat v2 (yang menang di TRAIN) ===
sr_source  sr_near_atr_mult sr_action  sr_strong_score_bonus  sr_min_atr_for_breakout  train_pf  train_n  test_pf  test_n  test_wr  test_netpnl  test_maxdd
     both               3.0      skip                    4.0                      1.5      0.63     1149     1.69     944    49.58      1742.25      -96.35
     both               3.0      skip                    4.0                      1.9      0.61      989     1.71     849    49.47      1688.46     -106.54
     both               3.0      skip                    4.0                      1.3      0.61     1260     1.65     998    48.50      1692.71     -123.15
     both               3.0      skip                    4.0                      1.7      0.61     1054     1.70     885    49.27      1714.60     -106.48
     both               3.0      skip                    4.0                      2.1      0.60      944     1.76     810    50.12      1741.88     -102.96
     b

## 5. Grid search: sumber S/R x threshold jarak x aksi (skip/adjust) x bonus skor kuat

In [9]:
import itertools
import time as _time

GRID = {
    "sr_source": ["h1", "m15", "both"],
    "sr_near_atr_mult": [0.5, 1.0, 1.5, 2.0, 3.0],
    "sr_action": ["skip", "adjust"],
    "sr_strong_score_bonus": [0.0, 2.0, 4.0],
}

combos = list(itertools.product(*GRID.values()))
print(f"Total kombinasi grid: {len(combos)}")

t0 = _time.time()
grid_results = []
for idx, combo in enumerate(combos):
    params = dict(zip(GRID.keys(), combo))
    trades = run_backtest_v28(df_train, **params)
    metrics = evaluate(trades, INITIAL_EQUITY)
    metrics.update(params)
    grid_results.append(metrics)
    if (idx + 1) % 20 == 0:
        print(f"  [{idx+1}/{len(combos)}] {_time.time()-t0:.0f}s")

grid_df = pd.DataFrame(grid_results)
print(f"\nGrid search selesai dalam {_time.time()-t0:.0f}s")

grid_valid = grid_df[grid_df["total_trades"] >= MIN_SAMPLE_TRAIN].sort_values("profit_factor", ascending=False)
print(f"\n=== Top 20 kandidat (sample TRAIN >= {MIN_SAMPLE_TRAIN}) ===")
print(grid_valid.head(20).to_string(index=False))

print(f"\nBaseline TRAIN: PF={baseline_train['profit_factor']}, net_pnl={baseline_train['net_pnl']}, n={baseline_train['total_trades']}")
beating = grid_valid[grid_valid["profit_factor"] > baseline_train["profit_factor"]]
print(f"Kandidat mengungguli baseline TRAIN (PF lebih tinggi): {len(beating)} dari {len(grid_valid)}")

Total kombinasi grid: 90


  [20/90] 18s


  [40/90] 36s


  [60/90] 54s


  [80/90] 72s



Grid search selesai dalam 81s

=== Top 20 kandidat (sample TRAIN >= 30) ===
 total_trades  win_rate_pct  profit_factor  net_pnl  max_drawdown_pct sr_source  sr_near_atr_mult sr_action  sr_strong_score_bonus
          783         19.80           0.55  -591.50           -591.50      both               3.0      skip                    4.0
          940         19.89           0.54  -724.63           -724.63      both               3.0      skip                    2.0
         1068         18.91           0.50  -905.22           -905.22      both               2.0      skip                    4.0
          983         19.84           0.50  -810.91           -810.91       m15               3.0      skip                    4.0
         1111         19.53           0.49  -937.52           -937.52       m15               3.0      skip                    2.0
         1393         18.52           0.49 -1196.07          -1196.07      both               1.5      skip                    2.0
      

## 6. Validasi TEST out-of-sample (kandidat yang mengungguli baseline TRAIN)

In [10]:
candidates_passing = beating.head(20)
print(f"Kandidat TRAIN mengungguli baseline: {len(candidates_passing)}")

if len(candidates_passing) == 0:
    print("\n>>> TIDAK ADA kandidat mengungguli baseline di TRAIN. Validasi TEST DIBATALKAN.")
else:
    test_results = []
    for _, row in candidates_passing.iterrows():
        params = {k: row[k] for k in GRID.keys()}
        trades_test = run_backtest_v28(df_test, **params)
        m_test = evaluate(trades_test, INITIAL_EQUITY)
        test_results.append({**params, "train_pf": row["profit_factor"], "train_n": row["total_trades"],
                              "test_pf": m_test["profit_factor"], "test_n": m_test["total_trades"],
                              "test_wr": m_test["win_rate_pct"], "test_netpnl": m_test["net_pnl"],
                              "test_maxdd": m_test["max_drawdown_pct"]})

    test_df = pd.DataFrame(test_results)
    print("\n=== Validasi TEST utk kandidat yang menang di TRAIN ===")
    print(test_df.to_string(index=False))

    print(f"\nBaseline TEST: PF={baseline_test['profit_factor']}, net_pnl={baseline_test['net_pnl']}, "
          f"max_dd={baseline_test['max_drawdown_pct']}, n={baseline_test['total_trades']}")

    robust = test_df[(test_df["test_pf"] > baseline_test["profit_factor"]) & (test_df["test_n"] >= MIN_SAMPLE_TEST)]
    print(f"\n>>> Kandidat ROBUST (unggul TRAIN & TEST vs baseline, sample TEST>={MIN_SAMPLE_TEST}): {len(robust)}")
    if len(robust) > 0:
        print(robust.to_string(index=False))

Kandidat TRAIN mengungguli baseline: 20



=== Validasi TEST utk kandidat yang menang di TRAIN ===
sr_source  sr_near_atr_mult sr_action  sr_strong_score_bonus  train_pf  train_n  test_pf  test_n  test_wr  test_netpnl  test_maxdd
     both               3.0      skip                    4.0      0.55      783     1.70     397    44.58       748.35     -117.50
     both               3.0      skip                    2.0      0.54      940     1.59     472    43.64       750.86     -172.52
     both               2.0      skip                    4.0      0.50     1068     1.78     548    46.90      1070.41     -121.15
      m15               3.0      skip                    4.0      0.50      983     1.73     498    46.39       892.67      -81.68
      m15               3.0      skip                    2.0      0.49     1111     1.62     568    45.42       890.15     -121.95
     both               1.5      skip                    2.0      0.49     1393     1.58     721    44.80      1065.31     -216.85
     both               1.

## 7. Ablation test tambahan: dampak filter S/R terpisah utk tiap tahun (robust lintas rezim 2019-2024?)

Sesuai batasan brief: filter harus robust lintas rezim (2019-2024 yang lebih merata), bukan cuma
2026 yang didominasi trending. Cek kandidat terbaik (kalau ada yang robust) breakdown per tahun.

In [11]:
if 'robust' in dir() and len(robust) > 0:
    best = robust.iloc[0]
    best_params = {k: best[k] for k in GRID.keys()}
    print(f"Kandidat terbaik utk breakdown tahunan: {best_params}")

    trades_full_baseline = run_backtest_v28(df, sr_source="none", use_fixed_lot=True, fixed_lot_value=0.03, initial_equity_override=2000.0)
    trades_full_filtered = run_backtest_v28(df, **best_params, use_fixed_lot=True, fixed_lot_value=0.03, initial_equity_override=2000.0)

    for trades, label in [(trades_full_baseline, "Baseline (tanpa filter S/R)"), (trades_full_filtered, "Dengan filter S/R")]:
        trades = trades.copy()
        trades["year"] = pd.to_datetime(trades["entry_time"]).dt.year
        print(f"\n=== {label} -- breakdown per tahun ===")
        for year, g in trades.groupby("year"):
            m = evaluate(g, 2000.0)
            print(f"  {year}: n={m['total_trades']}, win_rate={m['win_rate_pct']}%, PF={m['profit_factor']}, net_pnl=${m['net_pnl']}")
else:
    print("Tidak ada kandidat robust -- lewati breakdown tahunan (tidak relevan tanpa kandidat yang layak).")

Kandidat terbaik utk breakdown tahunan: {'sr_source': 'both', 'sr_near_atr_mult': np.float64(3.0), 'sr_action': 'skip', 'sr_strong_score_bonus': np.float64(4.0)}



=== Baseline (tanpa filter S/R) -- breakdown per tahun ===
  2019: n=473, win_rate=3.59%, PF=0.08, net_pnl=$-1725.69
  2020: n=422, win_rate=22.99%, PF=0.5, net_pnl=$-1244.91
  2021: n=430, win_rate=16.74%, PF=0.45, net_pnl=$-1210.13
  2022: n=454, win_rate=21.59%, PF=0.47, net_pnl=$-1248.21
  2023: n=503, win_rate=14.71%, PF=0.36, net_pnl=$-1615.89
  2024: n=402, win_rate=33.08%, PF=0.75, net_pnl=$-563.0
  2025: n=444, win_rate=49.32%, PF=1.47, net_pnl=$1368.26
  2026: n=259, win_rate=59.07%, PF=2.35, net_pnl=$3804.89

=== Dengan filter S/R -- breakdown per tahun ===
  2019: n=174, win_rate=3.45%, PF=0.06, net_pnl=$-745.08
  2020: n=152, win_rate=30.92%, PF=0.85, net_pnl=$-122.17
  2021: n=140, win_rate=21.43%, PF=0.7, net_pnl=$-211.99
  2022: n=137, win_rate=26.28%, PF=0.69, net_pnl=$-225.05
  2023: n=180, win_rate=20.0%, PF=0.46, net_pnl=$-470.2
  2024: n=131, win_rate=33.59%, PF=0.89, net_pnl=$-89.04
  2025: n=172, win_rate=42.44%, PF=1.05, net_pnl=$64.55
  2026: n=94, win_rate=63

## 8. Kesimpulan

*(diisi setelah lihat hasil eksekusi lengkap Section 3-7 -- placeholder)*

## 9. Validasi menyeluruh: v28 (filter S/R + syarat ATR breakout, kandidat v2 terbaik)
vs v13 murni -- data aktual, bukan cuma PF/win rate mentah

**Kandidat v2 yang dipakai** (PF TEST tertinggi di antara yg robust, Section 4d):
`sr_source="both", sr_near_atr_mult=3.0, sr_action="skip", sr_strong_score_bonus=4.0,
sr_min_atr_for_breakout=2.1`

**Metodologi validasi** (sama persis pola v14/v15/v27 yang sudah tervalidasi sebelumnya):
1. Perbandingan langsung metrik dasar (win rate, PF, net PnL, max DD) full period 2019-2026
2. **Monte Carlo** (v14-style): reshuffle urutan trade 10.000x, cek distribusi max drawdown --
   apakah v28 beneran lebih tahan risiko urutan, atau cuma kebetulan urutan historisnya bagus
3. **Deflated Sharpe Ratio** (v15-style): apakah v28 "menang" dari v13 itu signifikan scr
   statistik (bukan false discovery dari coba-coba banyak kombinasi grid search)
4. **Momentum exhaustion breakdown** (v27-style): apakah kualitas manajemen risiko (exit_reason,
   loss rate per momentum chain) juga membaik, bukan cuma angka agregat

In [12]:
BEST_V2_PARAMS = dict(
    sr_source="both", sr_near_atr_mult=3.0, sr_action="skip",
    sr_strong_score_bonus=4.0, sr_min_atr_for_breakout=2.1,
)

# Simulasi TANPA kill-switch (biar keliatan drawdown "mentah" apa adanya) -- TAPI basis
# lot & modal diubah spy angkanya gak meledak tak terhingga: lot FIXED 0.03 (bukan ikut
# equity), modal basis $2000 (bukan $100) sesuai instruksi user, spy simulasi 7.6 tahun
# tetap bisa dibaca angkanya (dolar & persen) meski gak direm kill-switch.
FULL_PERIOD_LOT = 0.03

FULL_PERIOD_EQUITY = 2000.0

trades_v13_full = run_backtest_v28(
    df, sr_source="none", use_fixed_lot=True,
    fixed_lot_value=FULL_PERIOD_LOT, initial_equity_override=FULL_PERIOD_EQUITY,
)

trades_v28_full = run_backtest_v28(
    df, **BEST_V2_PARAMS, use_fixed_lot=True,
    fixed_lot_value=FULL_PERIOD_LOT, initial_equity_override=FULL_PERIOD_EQUITY,
)

m_v13 = evaluate(trades_v13_full, FULL_PERIOD_EQUITY)
m_v28 = evaluate(trades_v28_full, FULL_PERIOD_EQUITY)

print("=== FULL PERIOD 2019-2026: v13 murni vs v28 (S/R + ATR breakout) ===")
compare_df = pd.DataFrame([
    {"strategy": "v13 (baseline)", **m_v13},
    {"strategy": "v28 (S/R+ATR filter)", **m_v28},
])
print(compare_df.to_string(index=False))

print(f"\nSelisih trade: {m_v28['total_trades'] - m_v13['total_trades']} "
      f"({(m_v28['total_trades']/m_v13['total_trades']-1)*100:+.1f}%)")
print(f"Selisih win rate: {m_v28['win_rate_pct'] - m_v13['win_rate_pct']:+.2f} poin persen")
print(f"Selisih PF: {m_v28['profit_factor'] - m_v13['profit_factor']:+.2f}")
print(f"Selisih net PnL: ${m_v28['net_pnl'] - m_v13['net_pnl']:+.2f} ({(m_v28['net_pnl']/m_v13['net_pnl']-1)*100:+.1f}% relatif)")
print(f"Selisih max DD: {m_v28['max_drawdown_pct'] - m_v13['max_drawdown_pct']:+.2f} poin persen")

=== FULL PERIOD 2019-2026: v13 murni vs v28 (S/R + ATR breakout) ===
            strategy  total_trades  win_rate_pct  profit_factor  net_pnl  max_drawdown_pct
      v13 (baseline)          3387         25.48           0.87 -2434.69           -386.97
v28 (S/R+ATR filter)          1754         35.63           1.25  2967.73           -116.84

Selisih trade: -1633 (-48.2%)
Selisih win rate: +10.15 poin persen
Selisih PF: +0.38
Selisih net PnL: $+5402.42 (-221.9% relatif)
Selisih max DD: +270.13 poin persen


### 9b. Breakdown per tahun (robust lintas rezim, bukan cuma menang di 1-2 tahun)

In [13]:
def yearly_breakdown(trades, label):
    trades = trades.copy()
    trades["year"] = pd.to_datetime(trades["entry_time"]).dt.year
    rows = []
    for year, g in trades.groupby("year"):
        m = evaluate(g, FULL_PERIOD_EQUITY)
        rows.append({"year": year, "strategy": label, **m})
    return pd.DataFrame(rows)

yearly_v13 = yearly_breakdown(trades_v13_full, "v13")
yearly_v28 = yearly_breakdown(trades_v28_full, "v28")
yearly_compare = pd.concat([yearly_v13, yearly_v28]).sort_values(["year", "strategy"])
print(yearly_compare[["year", "strategy", "total_trades", "win_rate_pct", "profit_factor", "net_pnl", "max_drawdown_pct"]].to_string(index=False))

years_v28_better = 0
years_total = 0
for year in sorted(yearly_v13["year"].unique()):
    pf_v13 = yearly_v13[yearly_v13["year"]==year]["profit_factor"].values
    pf_v28 = yearly_v28[yearly_v28["year"]==year]["profit_factor"].values
    if len(pf_v13) and len(pf_v28):
        years_total += 1
        if pf_v28[0] > pf_v13[0]:
            years_v28_better += 1
print(f"\nv28 mengungguli v13 di {years_v28_better} dari {years_total} tahun (PF lebih tinggi)")

 year strategy  total_trades  win_rate_pct  profit_factor  net_pnl  max_drawdown_pct
 2019      v13           473          3.59           0.08 -1725.69            -86.28
 2019      v28           176          5.11           0.08  -724.25            -36.21
 2020      v13           422         22.99           0.50 -1244.91           -149.01
 2020      v28           219         31.96           0.69  -438.09            -60.46
 2021      v13           430         16.74           0.45 -1210.13           -209.04
 2021      v28           169         23.08           0.72  -251.27            -71.34
 2022      v13           454         21.59           0.47 -1248.21           -271.45
 2022      v28           172         30.23           0.75  -236.09            -84.26
 2023      v13           503         14.71           0.36 -1615.89           -352.24
 2023      v28           208         23.56           0.64  -374.74           -101.22
 2024      v13           402         33.08           0.75  -563.0

## 10. Monte Carlo -- risiko urutan (sequence risk), v13 vs v28

Sama metodologi v14: reshuffle (permutation, bukan resample) urutan PnL trade 10.000x,
hitung ulang max drawdown tiap kali. Mengukur seberapa "beruntung" urutan historis, dan
seberapa DALAM drawdown bisa terjadi kalau urutannya sedikit berbeda -- utk v13 vs v28,
pakai basis lot FIXED (bukan compounding) spy adil dibandingkan skala PnL antar strategi.

In [14]:
def monte_carlo_drawdown(pnls: np.ndarray, initial_equity: float, n_sims: int = 10000, seed: int = 42) -> np.ndarray:
    rng = np.random.default_rng(seed)
    max_dds = np.empty(n_sims)
    for sim in range(n_sims):
        shuffled = rng.permutation(pnls)
        equity_curve = initial_equity + np.cumsum(shuffled)
        running_max = np.maximum.accumulate(np.concatenate([[initial_equity], equity_curve]))
        dd = (np.concatenate([[initial_equity], equity_curve]) - running_max) / running_max * 100
        max_dds[sim] = dd.min()
    return max_dds

pnls_v13 = trades_v13_full["pnl"].to_numpy()
pnls_v28 = trades_v28_full["pnl"].to_numpy()

mc_v13 = monte_carlo_drawdown(pnls_v13, FULL_PERIOD_EQUITY)
mc_v28 = monte_carlo_drawdown(pnls_v28, FULL_PERIOD_EQUITY)

actual_dd_v13 = m_v13["max_drawdown_pct"]
actual_dd_v28 = m_v28["max_drawdown_pct"]

print("=== Monte Carlo max drawdown (10.000 simulasi reshuffle) ===")
for label, mc, actual, n in [("v13", mc_v13, actual_dd_v13, len(pnls_v13)), ("v28", mc_v28, actual_dd_v28, len(pnls_v28))]:
    print(f"\n--- {label} (n={n} trade) ---")
    print(f"Actual historical max DD: {actual:.2f}%")
    print(f"MC median: {np.median(mc):.2f}%")
    print(f"MC P75: {np.percentile(mc, 25):.2f}%")
    print(f"MC P90: {np.percentile(mc, 10):.2f}%")
    print(f"MC P95: {np.percentile(mc, 5):.2f}%")
    print(f"MC P99: {np.percentile(mc, 1):.2f}%")
    print(f"MC worst: {mc.min():.2f}%")
    pct_worse_than_40 = (mc <= -40).mean() * 100
    print(f"%% simulasi breach kill-switch 40%%: {pct_worse_than_40:.2f}%")

print("\n=== Kesimpulan Monte Carlo ===")
print(f"v13 P95 drawdown: {np.percentile(mc_v13, 5):.2f}% | v28 P95 drawdown: {np.percentile(mc_v28, 5):.2f}%")
if np.percentile(mc_v28, 5) > np.percentile(mc_v13, 5):
    print("v28 SECARA RISIKO URUTAN lebih aman (P95 drawdown lebih dangkal) drpd v13.")
else:
    print("v13 SECARA RISIKO URUTAN lebih aman (P95 drawdown lebih dangkal) drpd v28.")

=== Monte Carlo max drawdown (10.000 simulasi reshuffle) ===

--- v13 (n=3387 trade) ---
Actual historical max DD: -386.97%
MC median: -126.00%
MC P75: -133.33%
MC P90: -141.91%
MC P95: -147.98%
MC P99: -160.81%
MC worst: -184.66%
%% simulasi breach kill-switch 40%%: 100.00%

--- v28 (n=1754 trade) ---
Actual historical max DD: -116.84%
MC median: -15.36%
MC P75: -19.12%
MC P90: -23.66%
MC P95: -27.27%
MC P99: -34.61%
MC worst: -54.70%
%% simulasi breach kill-switch 40%%: 0.24%

=== Kesimpulan Monte Carlo ===
v13 P95 drawdown: -147.98% | v28 P95 drawdown: -27.27%
v28 SECARA RISIKO URUTAN lebih aman (P95 drawdown lebih dangkal) drpd v13.


## 11. Deflated Sharpe Ratio -- apakah kemenangan v28 signifikan, bukan false discovery

Sama metodologi v15 (Bailey & Lopez de Prado 2014). v28 dipilih dari grid search besar
(90+108=198 kombinasi dicoba di Section 4c & 5) -- makin banyak kombinasi dicoba, makin
besar kemungkinan salah satu "menang" murni krn kebetulan cocok data historis. DSR
mengoreksi Sharpe Ratio v28 terhadap jumlah percobaan (N_TRIALS), panjang data, dan
skewness/kurtosis distribusi return.

In [15]:
from scipy.stats import norm, skew, kurtosis

def daily_returns_from_trades(trades: pd.DataFrame, initial_equity: float) -> pd.Series:
    trades = trades.copy()
    trades["entry_time"] = pd.to_datetime(trades["entry_time"])
    trades["date"] = trades["entry_time"].dt.date
    daily_pnl = trades.groupby("date")["pnl"].sum()
    daily_return = daily_pnl / initial_equity
    return daily_return

def sharpe_ratio(returns: pd.Series) -> float:
    if returns.std() == 0:
        return 0.0
    return returns.mean() / returns.std() * np.sqrt(252)

def expected_max_sharpe(n_trials: int, sr_std: float = 1.0) -> float:
    euler_gamma = 0.5772156649
    if n_trials <= 1:
        return 0.0
    return sr_std * ((1 - euler_gamma) * norm.ppf(1 - 1/n_trials) + euler_gamma * norm.ppf(1 - 1/(n_trials * np.e)))

def deflated_sharpe_ratio(sr: float, sr_benchmark: float, n_obs: int, skew_val: float, kurt_val: float) -> float:
    if n_obs <= 1:
        return 0.0
    denom = np.sqrt(1 - skew_val * sr + (kurt_val - 1) / 4 * sr**2)
    if denom <= 0:
        return 0.0
    psr = norm.cdf((sr - sr_benchmark) * np.sqrt(n_obs - 1) / denom)
    return psr

# Basis TEST period saja (2024-2026), konsisten dgn cara v15 mengukur -- out-of-sample murni
# NOTE: TEST period Sharpe/DSR pakai basis TRAIN/TEST section 3-6 ($100, lot dinamis) --
# BUKAN basis Section 9 ($2000, lot fixed) -- trades_base_test itu hasil dari Section 3.
returns_v13_test = daily_returns_from_trades(trades_base_test, INITIAL_EQUITY)
returns_v28_test_trades = run_backtest_v28(df_test, **BEST_V2_PARAMS)
returns_v28_test = daily_returns_from_trades(returns_v28_test_trades, INITIAL_EQUITY)

sr_v13 = sharpe_ratio(returns_v13_test)
sr_v28 = sharpe_ratio(returns_v28_test)
print(f"Sharpe Ratio (TEST period, annualized): v13={sr_v13:.4f}, v28={sr_v28:.4f}")

# N_TRIALS: total kombinasi dicoba di grid search v1 (90) + v2 (108) = 198
N_TRIALS = 198
sr_benchmark = expected_max_sharpe(N_TRIALS, sr_std=returns_v28_test.std() * np.sqrt(252) if returns_v28_test.std() > 0 else 1.0)
skew_v28 = skew(returns_v28_test) if len(returns_v28_test) > 2 else 0.0
kurt_v28 = kurtosis(returns_v28_test, fisher=False) if len(returns_v28_test) > 2 else 3.0
dsr_v28 = deflated_sharpe_ratio(sr_v28, sr_benchmark, len(returns_v28_test), skew_v28, kurt_v28)

print(f"\nN_TRIALS (kombinasi grid search dicoba): {N_TRIALS}")
print(f"SR benchmark (expected max dari {N_TRIALS} percobaan acak): {sr_benchmark:.4f}")
print(f"v28 Sharpe: {sr_v28:.4f}")
print(f"v28 Deflated Sharpe Ratio: {dsr_v28:.4f}")
if dsr_v28 >= 0.5:
    print(f"\n>>> SIGNIFIKAN (DSR>=0.5) -- v28 mengungguli v13 bukan krn kebetulan cocok grid search.")
else:
    print(f"\n>>> TIDAK signifikan (DSR<0.5) -- kemenangan v28 blm bisa dipastikan bukan false discovery dari 198 percobaan grid search.")

Sharpe Ratio (TEST period, annualized): v13=2.9506, v28=3.5152

N_TRIALS (kombinasi grid search dicoba): 198
SR benchmark (expected max dari 198 percobaan acak): 8.3518
v28 Sharpe: 3.5152
v28 Deflated Sharpe Ratio: 0.0000

>>> TIDAK signifikan (DSR<0.5) -- kemenangan v28 blm bisa dipastikan bukan false discovery dari 198 percobaan grid search.


## 12. Momentum exhaustion & kualitas manajemen risiko -- v13 vs v28

Sama metodologi v27: breakdown exit_reason & loss rate per momentum chain. Cek apakah
filter S/R v28 juga memperbaiki KUALITAS manajemen exit (bukan cuma volume/PF agregat).

In [16]:
for label, trades in [("v13", trades_v13_full), ("v28", trades_v28_full)]:
    print(f"=== {label}: exit_reason breakdown ===")
    wins = (trades["result"]=="WIN").sum()
    losses_n = (trades["result"]=="LOSS").sum()
    print(f"Total: {len(trades)} trade, WIN={wins} ({wins/len(trades)*100:.1f}%), LOSS={losses_n} ({losses_n/len(trades)*100:.1f}%)")
    if "mode" in trades.columns:
        print(trades["mode"].value_counts())
    print()

print("=== Ringkasan akhir: v13 (baseline) vs v28 (S/R + ATR breakout filter) ===")
summary_final = pd.DataFrame([
    {"metrik": "Total trade", "v13": m_v13["total_trades"], "v28": m_v28["total_trades"]},
    {"metrik": "Win rate %", "v13": m_v13["win_rate_pct"], "v28": m_v28["win_rate_pct"]},
    {"metrik": "Profit Factor", "v13": m_v13["profit_factor"], "v28": m_v28["profit_factor"]},
    {"metrik": "Net PnL ($)", "v13": m_v13["net_pnl"], "v28": m_v28["net_pnl"]},
    {"metrik": "Max Drawdown %", "v13": m_v13["max_drawdown_pct"], "v28": m_v28["max_drawdown_pct"]},
    {"metrik": "MC P95 Drawdown %", "v13": np.percentile(mc_v13, 5), "v28": np.percentile(mc_v28, 5)},
    {"metrik": "Sharpe Ratio (TEST)", "v13": sr_v13, "v28": sr_v28},
    {"metrik": "Deflated Sharpe Ratio (v28)", "v13": np.nan, "v28": dsr_v28},
    {"metrik": "Tahun v28 unggul PF", "v13": np.nan, "v28": f"{years_v28_better}/{years_total}"},
])
print(summary_final.to_string(index=False))

=== v13: exit_reason breakdown ===
Total: 3387 trade, WIN=863 (25.5%), LOSS=2524 (74.5%)
mode
NORMAL    3387
Name: count, dtype: int64

=== v28: exit_reason breakdown ===
Total: 1754 trade, WIN=625 (35.6%), LOSS=1129 (64.4%)
mode
NORMAL    1754
Name: count, dtype: int64

=== Ringkasan akhir: v13 (baseline) vs v28 (S/R + ATR breakout filter) ===
                     metrik          v13        v28
                Total trade  3387.000000       1754
                 Win rate %    25.480000      35.63
              Profit Factor     0.870000       1.25
                Net PnL ($) -2434.690000    2967.73
             Max Drawdown %  -386.970000    -116.84
          MC P95 Drawdown %  -147.982909 -27.267485
        Sharpe Ratio (TEST)     2.950581   3.515204
Deflated Sharpe Ratio (v28)          NaN        0.0
        Tahun v28 unggul PF          NaN        6/8


## 13. Kesimpulan akhir v28 vs v13

*(diisi setelah lihat hasil eksekusi lengkap Section 9-12)*